In [1]:
from woodf import WC

protocal = "http"  # 协议,一般是https(适用于正式上传),http(适用于本地脚本调试/开发)
domain = "wp.com"  # 域名
url = f"{protocal}://{domain}"  # 不要有多余的路径
consumer_key = "ck_d27091399219c406fb6f420f498aecfb8e6fe812"
consumer_secret = "cs_5bbfa3135dd9ff605f920f8c888b3036e0a69ec7"

wc = WC(
    api_consumer_key=consumer_key,
    api_consumer_secret=consumer_secret,
    api_url=url,
    timeout=30,
)
# 检查woo鉴权和链接(返回-1表示链接失败)
wc.get_product_count()
 

6

In [2]:
wc.fetch_existing_categories()

{'Bags': 29,
 'Cane': 40,
 'Hats': 19,
 'Man': 21,
 'Shoes': 30,
 'T-Shirts': 27,
 'Uncategorized': 15}

In [ ]:
import subprocess
import os
from typing import Optional, Dict


def download_image_with_curl(
    url: str,
    output_path: str,
    user_agent: str = "Mozilla/5.0",
    timeout: int = 30,
    silent: bool = False,
    extra_args: Optional[list] = None,
) -> bool:
    """
    使用系统 curl 命令下载图片（或其他文件）。

    参数:
        url (str): 要下载的文件 URL。
        output_path (str): 本地保存路径（含文件名）。
        user_agent (str): 请求头中的 User-Agent 字符串。
        timeout (int): 请求超时时间（秒）。
        silent (bool): 是否静默执行（不输出进度信息）。
        extra_args (Optional[list]): 其他要传给 curl 的额外参数列表。

    返回:
        bool: 下载成功返回 True，失败返回 False。

    抛出:
        FileNotFoundError: 如果 curl 不在系统 PATH 中。
        PermissionError: 如果没有写入目标路径的权限。
    """

    # 检查 curl 是否可用
    if not shutil.which("curl"):
        raise FileNotFoundError("curl 命令未找到，请确保已安装并添加到系统 PATH")

    # 确保输出目录存在
    output_dir = os.path.dirname(output_path)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # 构建 curl 命令参数
    cmd = ["curl", "-f", "--retry", "3", "--retry-delay", "5"]

    # 添加 User-Agent
    cmd += ["-A", user_agent]

    # 添加超时
    cmd += ["--max-time", str(timeout)]

    # 静默模式
    if silent:
        cmd += ["--silent"]
    else:
        cmd += ["--progress-bar"]

    # 输出文件
    cmd += ["-o", output_path]

    # 添加额外参数
    if extra_args:
        cmd += extra_args

    # 添加 URL 最后
    cmd += [url]

    try:
        print(f"正在下载: {url}")
        subprocess.run(cmd, check=True)
        print(f"文件已保存至: {output_path}")
        return True
    except subprocess.CalledProcessError as e:
        print(f"curl 执行失败，错误码: {e.returncode}")
        return False
    except PermissionError as pe:
        raise PermissionError(f"无权写入路径: {output_path}") from pe